<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/Lessons/Module_7/Lesson_7_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> Lesson 2.6 (Neural Networks Continued) </b> </font>

---

<font size = 5> <b> Notebook Index </b> </font>

1. [Learning Outcomes](#Learning-Outcomes)

2. [Introduction](#Introduction)

$
\newcommand{\lsum}{\displaystyle \sum\limits_{i=1}^{N}}
\newcommand{\parens}[1]{\left(#1\right)}
\newcommand{\dsfrac}[2]{\displaystyle\frac{#1}{#2}}
\newcommand{\dpfrac}[2]{\displaystyle\parens{\frac{#1}{#2}}}
\newcommand{\parderiv}[2]{\dsfrac{\partial #1}{\partial #2}}
\newcommand{\spc}{\hspace{0.1 pc}}
\newcommand{\ra}{\Rightarrow}
\newcommand{\of}[1]{{\scriptsize (#1)}}
\newcommand{\rule}{\Huge \hspace{-0.2 pc} \displaystyle\frac{\hspace{20 pc}}{\hspace{20 pc}}}
\newcommand{\mps}{\spc \frac{\textrm{m}}{\textrm{s}}}
\newcommand{\mpss}{\spc \frac{\textrm{m}}{\textrm{s}^2}}
$

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 1. Learning Outcomes </b> </font>

---

<font size = 5> <b> Learning Outcomes: </b> </font>

By the end of this lesson, students will be able to:

  1. <b>Build</b> binary and multi-class neural networks using Keras
  2. <b>Explain</b> sigmoid vs softmax activation
  3. <b>Apply</b> feature scaling and one-hot encoding
  4. <b>Use</b> train/test splits to evaluate generalization
  5. <b>Interpret</b> training vs validation curves
  6. <b>Diagnose</b> overfitting
  7. <b>Compare</b> shallow vs wider architectures


[Return to Top](#Notebook-Start)

<a name="Introduction"></a>

---

#<font size = 6> <b> 2. Introduction </b> </font>

---

---

##<font size = 5> <b> 2.2 Prepare the Program </b> </font>

### <b> Import Libraries </b>

In [ ]:
##=============================================================================================##
## Import Libraries:                                                                           ##
##=============================================================================================##

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from IPython.display import display_html

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline

from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import plot_model

import warnings
warnings.filterwarnings('ignore')

### <b> Define Useful Functions </b>

In [ ]:
#@title This cell defines the functions: display_dataframes

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  dataframe_list - List of DataFrames to be displayed                              ##
##            title_list     - List of titles for the displayed DataFrames                     ##
##            n_items        - Number of items to display (optional, default = 5)              ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(dataframe_list, title_list, n_items = 5, tail = False):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ""

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(dataframe_list, title_list):

    # Convert the current dataframe info to html:

    if (tail == True):

      html_df = pd.DataFrame(df).tail(n_items).to_html()

    else:

      html_df = pd.DataFrame(df).head(n_items).to_html()

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

    html_str += "<div style='display: inline-block; margin-right: 20px; vertical-align: top;'>"

    html_str += "<h3 style='text-align: center;'>" + str(title) + "</h3><hr>" + str(html_df)

    html_str += "</div>"

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

### <b> Load the Data </b>

In [ ]:
##=============================================================================================##
## Load Titanic Passenger Data and Parse the Features and Target Data:                         ##
##=============================================================================================##

# Load and clean the Titanic data:

titanic = sns.load_dataset('titanic').dropna().reset_index().drop('index', axis = 1)

# Set the target data (1 = survived, 0 = did not survive):

y = titanic['survived'].values

# Set the feature data (extract the target data):

X = titanic.drop('survived', axis = 1)

# Split the data into testing and training subsets:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Reset the index for testing and training data subsets:

X_train = X_train.reset_index().drop('index', axis = 1)
X_test  = X_test.reset_index().drop('index', axis = 1)

# Display the feature and target data:

display_dataframes([X_train, y_train], ["Training Features (X_train)", "Training Targets (y_train)"])

display_dataframes([X_test, y_test], ["Testing Features (X_test)", "Testing Targets (y_test)"])

[Return to Top](#Notebook-Start)

<a name="Introduction"></a>

---

#<font size = 6> <b> 3. Creating Neural Networks Using Keras Library </b> </font>

---

In this section, we focuse on using the keras library to build an Artificial Neural Network for the titanic dataset. We now use all the rows of the data and the age and fare columns to build a basic network. After building the model we will visualize the loss by epoch.

---

##<font size = 5> <b> 3.1 Simple Neural Network Example </b> </font>

To begin, we use `keras` and the `Sequential` model to create a neural network with the following architecture:

- 1 Hidden Layer
- 1 Node in Hidden Layer
- 1 output node
- Sigmoid activation function on all nodes

This is the equivalent to the architecture of our neural network from the previous lesson.

### <b> Set the Feature Data </b>

In [ ]:
##=============================================================================================##
## Set the feature data (passenger age and ticket price):                                      ##
##=============================================================================================##

X_simple_train = X_train[['age', 'fare']].values

X_simple_test  = X_test[['age', 'fare']].values

### <b> Create the Neural Network Model </b>

In [ ]:
##=============================================================================================##
## Create a Neural Network With One Node in One Hidden Layer:                                  ##
##=============================================================================================##

simple_model = Sequential([
    Dense(1, activation = 'sigmoid'),
    Dense(1, activation = 'sigmoid')
    ])

### <b> Specify Parameters for Neural Network </b>

Now, we use our `single_node_model` and specify the following elements using the .compile method.

* optimizer = rmsprop
* loss = binary_crossentropy or bce
* metrics = ['accuracy']

<br>

> rmpsprop = Root Mean Square Propagation

<br>

> Binary cross entropy is a loss function used for binary classification models that quantifies the difference between the predicted probabilities and the true binary labels (0 or 1)


In [ ]:
##=============================================================================================##
## Set the Parameters of the Neural Network:                                                   ##
##=============================================================================================##

simple_model.compile(optimizer = 'rmsprop',
                     loss      = 'binary_crossentropy',
                     metrics   = ['accuracy'])

### <b> Train (Fit) the Neural Network on the Titanic Data </b>

With our model setup, we fit the model below using the following parameters:

- `epochs = 20`
- `batch_size = 10`
- `verbose = 0`

In [ ]:
##=============================================================================================##
## Fit the Neural Network to the Titanic Data:                                                 ##
##=============================================================================================##

history_simple = simple_model.fit(x = X_simple_train, y = y_train, epochs = 20, batch_size = 10, verbose = 0)

### <b> Evaluate the Neural Network's Performance </b>

Now, we use the .evaluate method of our single_node_model with the X and y arrays to examine the loss and accuracy of the model.

In [ ]:
##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

results = simple_model.evaluate(X_simple_test, y_test)

simple_loss = results[0]
simple_acc = results[1]

print()
print("Simple Model:\n")
print(f"Loss: {simple_loss:.4f}")
print(f"Accuracy: {simple_acc:.4f}")
print()

##=============================================================================================##
## Plot the Accuracy as a Function of Training Epoch:                                          ##
##=============================================================================================##

plt.figure(figsize = (10, 6))

simple_plot = plt.plot(history_simple.history['accuracy'])


---

##<font size = 5> <b> 3.2 A More Complex Model Example </b> </font>

To try to improve the model, we now build and evaluate a second model that uses a single hidden layer with 100 nodes, and a single output layer.  

For the hidden layer, we use the `relu` activation function and for the output layer use the `sigmoid` activation.

In [ ]:
##=============================================================================================##
## Create a Neural Network With One Hundred Nodes in One Hidden Layer:                         ##
##=============================================================================================##

# Set the number of nodes for each layer:

n_hidden_nodes = 100

n_output_nodes = 1

# Set the activation functions for each layer:

hidden_activation = 'relu'

output_activation = 'sigmoid'

# Create the Neural_Network:

complex_model = Sequential([
    Dense(n_hidden_nodes, activation = hidden_activation),
    Dense(n_output_nodes, activation = output_activation)
])

##=============================================================================================##
## Set the Parameters of the Neural Network:                                                   ##
##=============================================================================================##

complex_model.compile(optimizer = 'rmsprop',
                      loss      = 'bce',
                      metrics   = ['accuracy'])

##=============================================================================================##
## Fit the Neural Network to the Titanic Data:                                                 ##
##=============================================================================================##

history_complex = complex_model.fit(X_simple_train, y_train, epochs = 20, batch_size = 10, verbose = 0)

##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

results = complex_model.evaluate(X_simple_test, y_test)

complex_loss = results[0]
complex_acc  = results[1]

print()
print("Complex Model:\n")
print(f"Loss: {complex_loss:.4f}")
print(f"Accuracy: {complex_acc:.4f}")
print()

##=============================================================================================##
## Plot the Accuracy as a Function of Training Epoch:                                          ##
##=============================================================================================##

plt.figure(figsize = (10, 6))

complex_plot = plt.plot(history_complex.history['accuracy'])

[Return to Top](#Notebook-Start)

---

##<font size = 5> <b> 3.3 Importance of Data Preparation </b> </font>

Next, we focus on using the full dataset and remaining features with a single layered neural network. In the last example with only two features an accuracy of roughly 65% was achieved. Using more features and a similar network architecture you will see if the model improves.

### <b> Expanding the Feature Data </b>

For this section we use columns `['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class']` as our features to predict the target `survived`.

In [ ]:
##=============================================================================================##
## Set the Feature Data:                                                                       ##
##=============================================================================================##

X_full_train = X_train[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class']]

X_full_test  = X_test[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class']]

### <b> Preparing the Features </b>

Below, we use the `make_column_transformer` to prepare the features.  We use the `OneHotEncoder` with `drop = 'if_binary'` on all categorical features, and use the `StandardScaler` on the remaining features.  We assign the transformed data array to `X_t` below.

In [ ]:
##=============================================================================================##
## Transform the Data:                                                                         ##
##=============================================================================================##

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = ['sex', 'embarked', 'class']

t = make_column_transformer(
    (OneHotEncoder(drop='if_binary'), cat_cols),
    (StandardScaler(), num_cols)
)

X_t_train = t.fit_transform(X_full_train)

X_t_test = t.transform(X_full_test)

### <b> Create the Model </b>

In [ ]:
##=============================================================================================##
## Create a Neural Network With One Hundred Nodes in One Hidden Layer:                         ##
##=============================================================================================##

# Set the number of nodes for each layer:

n_hidden_nodes = 100

n_output_nodes = 1

# Set the activation functions for each layer:

hidden_activation = 'relu'

output_activation = 'sigmoid'

# Create the Neural_Network:

complex_model = Sequential([
    Dense(n_hidden_nodes, activation = hidden_activation),
    Dense(n_output_nodes, activation = output_activation)
])

##=============================================================================================##
## Set the Parameters of the Neural Network:                                                   ##
##=============================================================================================##

complex_model.compile(optimizer = 'rmsprop',
                      loss      = 'bce',
                      metrics   = ['accuracy'])

##=============================================================================================##
## Fit the Neural Network to the Titanic Data:                                                 ##
##=============================================================================================##

history_complex = complex_model.fit(X_t_train, y_train, epochs = 20, batch_size = 10, verbose = 0)

##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

results = complex_model.evaluate(X_t_test, y_test)

complex_loss = results[0]
complex_acc  = results[1]

print()
print("Complex Model:\n")
print(f"Loss: {complex_loss:.4f}")
print(f"Accuracy: {complex_acc:.4f}")
print()

##=============================================================================================##
## Plot the Accuracy as a Function of Training Epoch:                                          ##
##=============================================================================================##

plt.figure(figsize = (10, 6))

complex_plot = plt.plot(history_complex.history['accuracy'])

<a name="Introduction"></a>

---

#<font size = 6> <b> 4. Multi-Class Classification with keras </b> </font>

---

In this section, we focus on using keras to build a multi-class classification model using a wine dataset. We will use the version of the dataset loaded with scikitlearn. Below, the data is loaded and prepared. Rather than creating a train and test set, we will use the validation_split argument of the .fit function.


---

##<font size = 5> <b> 4.1 Prepare the Data </b> </font>

### <b> Load the Data </b>

In [ ]:
##=============================================================================================##
## Load the Data and Parse the Features and Target Data:                                       ##
##=============================================================================================##

# Load and clean the Titanic data:

wine = load_wine(as_frame = True)

# Set the target and feature data:

X, y = wine.data, wine.target

# Split the data into testing and training subsets:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Reset the index for testing and training data subsets:

X_train = X_train.reset_index().drop('index', axis = 1)
X_test  = X_test.reset_index().drop('index', axis = 1)

# Display the feature and target data:

display_dataframes([X_train, y_train], ["Training Features (X_train)", "Training Targets (y_train)"])

display_dataframes([X_test, y_test], ["Testing Features (X_test)", "Testing Targets (y_test)"])

### <b> Prepare the Data </b>

Because we are solving a multi-class classification problem, two things will change from the binary case.  First, we will need to present the network with a one hot encoded version of the *target*.  This can be accomplished using the `to_categorical` function. We build the one hot encoded version of the target and assign as an array to `y_ohe` below.


In [ ]:
##=============================================================================================##
## Convert the Categorical Data to a One Hot Encoded Version:                                  ##
##=============================================================================================##

y_ohe_train = to_categorical(y_train)

y_ohe_test = to_categorical(y_test)


---

##<font size = 5> <b> 4.2 Build the Model </b> </font>

To try to improve the model, we now build and evaluate a second model that uses a single hidden layer with 100 nodes, and a single output layer.  

For the hidden layer, we use the `relu` activation function and for the output layer use the `sigmoid` activation.

In [ ]:
##=============================================================================================##
## Create a Neural Network With One Hundred Nodes in One Hidden Layer:                         ##
##=============================================================================================##

# Set the number of nodes for each layer:

n_hidden_nodes = 100

n_output_nodes = 3

# Set the activation functions for each layer:

hidden_activation = 'relu'

output_activation = 'sigmoid'

# Create the Neural_Network:

classification_model = Sequential([
    Dense(n_hidden_nodes, activation = hidden_activation),
    Dense(n_output_nodes, activation = output_activation)
])

### <b> Compiling the Model </b>

For the compilation, rather than binary crossentropy as a loss function, we use `categorical_crossentropy`. We continue to use `accuracy` as the metric.

In [ ]:
##=============================================================================================##
## Set the Parameters of the Neural Network:                                                   ##
##=============================================================================================##

classification_model.compile(loss = 'categorical_crossentropy',
                             metrics   = ['accuracy'])

### <b> Fitting the Model </b>

Now, we fit the model using the following settings:

- `epochs = 100`
- `validation_split = 0.2`
- `verbose = 0`

In [ ]:
##=============================================================================================##
## Fit the Neural Network to the Titanic Data:                                                 ##
##=============================================================================================##

history_class = classification_model.fit(X_train, y_ohe_train, validation_split=0.2, epochs = 100, verbose = 0)

### <b> Evaluate the Neural Network's Performance </b>

In [ ]:
##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

results = classification_model.evaluate(X_test, y_ohe_test)

classification_loss = results[0]
classification_acc  = results[1]

print()
print("Complex Model:\n")
print(f"Loss: {complex_loss:.4f}")
print(f"Accuracy: {complex_acc:.4f}")
print()

##=============================================================================================##
## Plot the Accuracy as a Function of Training Epoch:                                          ##
##=============================================================================================##

plt.figure(figsize = (10, 6))

classification_plot = plt.plot(history_class.history['accuracy'])

---

##<font size = 5> <b> 4.4 Normalize the Feature Data </b> </font>

In [ ]:
##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

X_norm = StandardScaler().fit_transform(X)

# Split the data into testing and training subsets:

X_train_norm, X_test_norm, y_train, y_test = train_test_split(X_norm, y, test_size = 0.2, random_state = 42)

# Reset the index for testing and training data subsets:

X_train = X_train.reset_index().drop('index', axis = 1)
X_test  = X_test.reset_index().drop('index', axis = 1)

### <b> Re-Fit and Evaluate the Model </b>

In [ ]:
##=============================================================================================##
## Fit the Neural Network to the Titanic Data:                                                 ##
##=============================================================================================##

history_class = classification_model.fit(X_train_norm, y_ohe_train, validation_split=0.2, epochs = 100, verbose = 0)

##=============================================================================================##
## Check the Results of the Trained Neural Network:                                            ##
##=============================================================================================##

results = classification_model.evaluate(X_test_norm, y_ohe_test)

classification_loss = results[0]
classification_acc  = results[1]

print()
print("Complex Model:\n")
print(f"Loss: {complex_loss:.4f}")
print(f"Accuracy: {complex_acc:.4f}")
print()

##=============================================================================================##
## Plot the Accuracy as a Function of Training Epoch:                                          ##
##=============================================================================================##

plt.figure(figsize = (10, 6))

classification_plot = plt.plot(history_class.history['accuracy'])